In [3]:
import glob
import rasterio
from rasterio.merge import merge
import geopandas as gpd
from rasterio.mask import mask

# Mosaic

In [4]:
#Gather all individual country binary rasters
file_list = glob.glob(r"C:\FAO_commodities_presence_absence\raw_ee\GLC_30m_2022\*.tif")
src_files_to_mosaic = [rasterio.open(fp) for fp in file_list]
# 2. Merge rasters spatially into a single regional raster
# Using method='max' ensures that if country borders overlap, presence (1) overrides absence (0)
mosaic, out_transform = merge(
    src_files_to_mosaic, 
    method='max', 
    nodata=0
)
# 3. Update metadata to reflect the expanded regional extent and grid resolution
out_meta = src_files_to_mosaic[0].meta.copy()
out_meta.update({
    "driver": "GTiff",
    "height": mosaic.shape[1],
    "width": mosaic.shape[2],
    "transform": out_transform,
    "nodata": 0,
    "compress": "lzw"
})
with rasterio.open(r"C:\FAO_commodities_presence_absence\raw_ee\GLC_30m_2022\GLC_30m_2022.tif", "w", **out_meta) as dest:
    dest.write(mosaic)
#Close open file connections
for src in src_files_to_mosaic:
    src.close()

# Clip

In [3]:
# 1) Read AOI shapefile
aoi = gpd.read_file(r"C:\Data Spasial\FAO_Binary_Commodities_FPDP\INDONESIA\INDO_AOI.shp")

# 2) Make sure CRS matches the raster
# Use the mosaic file's CRS or the raster you created
with rasterio.open(r"C:\FAO_commodities_presence_absence\intermediate_process\rubber\alt_Mosaic_rubber_2020.tif") as src:
    raster_crs = src.crs
    print("Raster CRS:", raster_crs)

aoi = aoi.to_crs(raster_crs)

# 3) Clip the raster to AOI geometry
with rasterio.open(r"C:\FAO_commodities_presence_absence\intermediate_process\rubber\alt_Mosaic_rubber_2020.tif") as src:
    shapes = [geom for geom in aoi.geometry]
    clipped, out_transform = mask(
        src,
        shapes=shapes,
        crop=True,
        nodata=0,
        all_touched=False
    )

    out_meta = src.meta.copy()
    out_meta.update({
        "driver": "GTiff",
        "height": clipped.shape[1],
        "width": clipped.shape[2],
        "transform": out_transform,
        "nodata": 0,
        "compress": "lzw"
    })

# 4) Save clipped output
with rasterio.open(
    r"C:\FAO_commodities_presence_absence\intermediate_process\rubber\alt_Mosaic_rubber_2020_clipped.tif",
    "w",
    **out_meta
) as dest:
    dest.write(clipped)

Raster CRS: EPSG:4326


# Reduce Scale

In [ ]:
import argparse
from pathlib import Path
import numpy as np
import rasterio
from rasterio.enums import Resampling
def downsample_raster(
    src_path: Path,
    dst_path: Path,
    factor: int = 10,
    resampling: Resampling = Resampling.average,
    nodata: float | None = None,
) -> np.ndarray:
    """
    Downsample a single raster by an integer factor using a decimated read.
    """
    with rasterio.open(src_path) as src:
        new_width = max(1, src.width // factor)
        new_height = max(1, src.height // factor)
 
        data = src.read(
            out_shape=(src.count, new_height, new_width),
            resampling=resampling,)
 
        new_transform = src.transform * src.transform.scale(
            src.width / new_width, src.height / new_height)
 
        profile = src.profile.copy()
        profile.update(
            {
                "height": new_height,
                "width": new_width,
                "transform": new_transform,
                "dtype": data.dtype,
                "nodata": nodata if nodata is not None else src.nodata,
                "compress": "lzw",
            }
        )
 
        dst_path.parent.mkdir(parents=True, exist_ok=True)
        with rasterio.open(dst_path, "w", **profile) as dst:
            dst.write(data)
 
    return data

In [ ]:
# Remap Esri land-cover values, reclassify, and create binary rasters
from pathlib import Path

import numpy as np
import rasterio

input_path = Path(r"C:\FAO_commodities_presence_absence\raw_ee\Esri_LC_2025\Esri_LandCov_2025.tif")
output_directory = input_path.parent
remapped_output_path = output_directory / "Esri_LandCov_2025_remapped.tif"
output_path = output_directory / "Esri_LandCov_2025_reclassified.tif"

# Original Esri values are remapped to the paper's sequential class values.
remap_from = [1, 2, 4, 5, 7, 8, 9, 10, 11]
remap_to = [1, 2, 3, 4, 5, 6, 7, 8, 9]
remap_mapping = dict(zip(remap_from, remap_to))

#After remapping: 5 = Built up, 4 = Cropland, 9 = Rangeland.
#Esri Land Cover reclassification
#class_mapping = {5: 1, 4: 2, 9: 3}
#binary_outputs = {
#    "built_up": (5, output_directory / "Esri_LandCov_2025_built_up_binary.tif"),
#    "cropland": (4, output_directory / "Esri_LandCov_2025_cropland_binary.tif"),
#    "rangeland": (9, output_directory / "Esri_LandCov_2025_rangeland_binary.tif"),}

class_mapping = {
    1: 1, 11: 1, 12: 1, 20: 1,   # Cropland
    2: 2,                       # Built-up
    3: 3, 121: 3, 122: 3,       # Shrubland
    4: 4,                       # Grassland
}
binary_outputs = {
    "cropland": (1, output_directory / "GLCFS_LandCov_2020_cropland_binary.tif"),
    "built_up": (2, output_directory / "GLCFS_LandCov_2020_built_up_binary.tif"),
    "shrubland": (3, output_directory / "GLCFS_LandCov_2020_shrubland_binary.tif"),
    "grassland": (4, output_directory / "GLCFS_LandCov_2020_grassland_binary.tif"),
}
with rasterio.open(input_path) as src:
    source = src.read(1)
    profile = src.profile.copy()
    source_nodata = src.nodata

    # Use 0 for source values not present in remap_from; these become Other.
    remapped = np.zeros(source.shape, dtype=np.uint8)
    for source_value, remapped_value in remap_mapping.items():
        remapped[source == source_value] = remapped_value

    output_directory.mkdir(parents=True, exist_ok=True)

    # Save the intermediate raster produced by the remapping step.
    remapped_profile = profile.copy()
    remapped_profile.update(dtype="uint8", count=1, nodata=0, compress="lzw")
    with rasterio.open(remapped_output_path, "w", **remapped_profile) as dst:
        dst.write(remapped, 1)

    reclassified = np.full(source.shape, 4, dtype=np.uint8)
    for remapped_value, output_value in class_mapping.items():
        reclassified[remapped == remapped_value] = output_value

    # Keep nodata pixels separate from the "Other" class when nodata exists.
    if source_nodata is not None:
        nodata_mask = source == source_nodata
        reclassified[nodata_mask] = 0
        profile["nodata"] = 0
    else:
        nodata_mask = np.zeros(source.shape, dtype=bool)
        profile["nodata"] = None

    profile.update(dtype="uint8", count=1, compress="lzw")

    with rasterio.open(output_path, "w", **profile) as dst:
        dst.write(reclassified, 1)

    # Binary rasters use 1 for the selected remapped class and 0 elsewhere.
    binary_profile = profile.copy()
    binary_profile["nodata"] = None
    for class_name, (remapped_value, binary_path) in binary_outputs.items():
        binary = (remapped == remapped_value).astype(np.uint8)
        binary[nodata_mask] = 0
        with rasterio.open(binary_path, "w", **binary_profile) as dst:
            dst.write(binary, 1)
        print(f"{class_name.title()} binary raster written to: {binary_path}")

print(f"Remapped raster written to: {remapped_output_path}")
print(f"Reclassified raster written to: {output_path}")

Built_Up binary raster written to: C:\FAO_commodities_presence_absence\raw_ee\Esri_LC_2025\Esri_LandCov_2025_built_up_binary.tif
Cropland binary raster written to: C:\FAO_commodities_presence_absence\raw_ee\Esri_LC_2025\Esri_LandCov_2025_cropland_binary.tif
Rangeland binary raster written to: C:\FAO_commodities_presence_absence\raw_ee\Esri_LC_2025\Esri_LandCov_2025_rangeland_binary.tif
Remapped raster written to: C:\FAO_commodities_presence_absence\raw_ee\Esri_LC_2025\Esri_LandCov_2025_remapped.tif
Reclassified raster written to: C:\FAO_commodities_presence_absence\raw_ee\Esri_LC_2025\Esri_LandCov_2025_reclassified.tif


In [ ]:
from pathlib import Path
import numpy as np
import rasterio

input_path = Path(r"C:\FAO_commodities_presence_absence\raw_ee\GLC_30m_2022\GLC_30m_2022.tif")
output_directory = input_path.parent
output_path = output_directory / "GLC_30m_2022_reclassified.tif"
# 1: Cropland
# 2: Built-up Land
# 3: Shrubland
# 4: Grassland
class_mapping = {
    10: 1, 
    11: 1,
    12: 1, 
    20: 1,   
    190: 2,                
    120: 3, 
    121: 3, 
    122: 3,
    130: 4,               
}
binary_outputs = {
    "cropland": (1, output_directory / "GLC_30m_2022_cropland_binary.tif"),
    "built_up": (2, output_directory / "GLC_30m_2022_built_up_binary.tif"),
    "shrubland": (3, output_directory / "GLC_30m_2022_shrubland_binary.tif"),
    "grassland": (4, output_directory / "GLC_30m_2022_grassland_binary.tif"),
}
with rasterio.open(input_path) as src:
    source = src.read(1)
    profile = src.profile.copy()
    source_nodata = src.nodata

    output_directory.mkdir(parents=True, exist_ok=True)

    # Unmapped source values remain 0 (Other/undefined).
    reclassified = np.zeros(source.shape, dtype=np.uint8)
    for source_value, output_value in class_mapping.items():
        reclassified[source == source_value] = output_value

    if source_nodata is not None:
        nodata_mask = source == source_nodata
        reclassified[nodata_mask] = 0
        profile["nodata"] = 0
    else:
        nodata_mask = np.zeros(source.shape, dtype=bool)
        profile["nodata"] = None

    profile.update(dtype="uint8", count=1, compress="lzw")

    with rasterio.open(output_path, "w", **profile) as dst:
        dst.write(reclassified, 1)

    binary_profile = profile.copy()
    binary_profile["nodata"] = None
    for class_name, (output_value, binary_path) in binary_outputs.items():
        binary = (reclassified == output_value).astype(np.uint8)
        binary[nodata_mask] = 0
        with rasterio.open(binary_path, "w", **binary_profile) as dst:
            dst.write(binary, 1)
        print(f"{class_name.title()} binary raster written to: {binary_path}")

print(f"Reclassified raster written to: {output_path}")

Cropland binary raster written to: C:\FAO_commodities_presence_absence\raw_ee\GLC_30m_2022\GLC_30m_2022_cropland_binary.tif
Built_Up binary raster written to: C:\FAO_commodities_presence_absence\raw_ee\GLC_30m_2022\GLC_30m_2022_built_up_binary.tif
Shrubland binary raster written to: C:\FAO_commodities_presence_absence\raw_ee\GLC_30m_2022\GLC_30m_2022_shrubland_binary.tif
Grassland binary raster written to: C:\FAO_commodities_presence_absence\raw_ee\GLC_30m_2022\GLC_30m_2022_grassland_binary.tif
Reclassified raster written to: C:\FAO_commodities_presence_absence\raw_ee\GLC_30m_2022\GLC_30m_2022_reclassified.tif
